# CCPD — Data and preprocessing

Completed dataset: **dataset/**. Source data: **source_data/**.
This notebook reproduces the preparation of 5,000 unique CCPD2019 images.
Run all cells from this folder. Existing datasets are preserved; another run
creates dataset_run_2/, dataset_run_3/, etc.

Outputs: image-level train/validation/test splits, size groups, three scales,
YOLO annotations, ground-truth plate crops, OCR targets and preprocessing.
Model training and evaluation are performed separately.

Split: 3,500 / 700 / 800. Eight duplicate images in the initial selection were
replaced within the same size and angle groups; selection records are included.


## Step 0 — Settings

Run this notebook from this folder. All input paths are relative to it.
Keep the default sample size and seed for the team dataset. A repeated full run
uses a new numbered output folder rather than replacing existing results.


In [ ]:
from pathlib import Path

WORK_DIR = Path.cwd().resolve()
if not (WORK_DIR/'project_config.json').is_file():
    raise FileNotFoundError('Open Data_Preprocessing.ipynb from the CCPD_Data_Preprocessing folder or set WORK_DIR to that folder.')
DATASET_DIR = WORK_DIR/'source_data'/'images'
CCPD_ROOT = DATASET_DIR
OUTPUT_ROOT = WORK_DIR/'dataset'
output_number = 2
while OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
    OUTPUT_ROOT = WORK_DIR/f'dataset_run_{output_number}'
    output_number += 1

SAMPLE_N = 5000
RANDOM_SEED = 42
SPLIT_RATIOS = {'train':0.70,'val':0.14,'test':0.16}
SCALES = (1.0,0.75,0.5)

assert SAMPLE_N > 0
assert abs(sum(SPLIT_RATIOS.values())-1.0)<1e-10
assert 1.0 in SCALES and all(0<s<=1.0 for s in SCALES)

print('Input folder:',DATASET_DIR)
print('Output folder:',OUTPUT_ROOT)
print('Original images:',SAMPLE_N)


## Step 1 — Check the included input data

All selected images are already included. This step checks the completion receipt
and exact image count. No downloading or full-dataset extraction is performed.
Actual source checksums are verified in Step 3 before splitting.


In [ ]:
import importlib.util
import os
import subprocess
import sys
import hashlib
import json
import shutil
import time

if importlib.util.find_spec('tqdm') is None:
    raise ImportError('Install requirements.txt in this notebook kernel environment first.')
from tqdm.auto import tqdm


In [ ]:
receipt_path = WORK_DIR/'source_data'/'subset_complete.json'
if not receipt_path.is_file() or not DATASET_DIR.is_dir():
    raise FileNotFoundError('The package is incomplete: source_data/images and source_data/subset_complete.json are required.')

source_receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
input_paths = [p for p in DATASET_DIR.rglob('*') if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.png')]
assert len(input_paths)==SAMPLE_N==len(source_receipt['images']), 'Incomplete or unexpected input dataset'

expected_hashes = {Path(r['image_relpath']).relative_to('images').as_posix():r['sha256'] for r in source_receipt['images']}
print(f'Ready: {len(input_paths):,} included source images. No download required.')


## Step 2 — Install libraries and understand CCPD annotations

CCPD2019 records annotations in each image filename. For example, a standard filename has these fields:

`area-tilt-bounding_box-four_corners-plate_characters-brightness-blur.jpg`

We use the **box** for object detection, **plate characters** for OCR, and **angle metadata** for grouping images by difficulty. No manual labeling is required for correctly named CCPD2019 images.

In [ ]:
# Install missing packages only if needed, then import the processing libraries.
import csv
import hashlib
import json
import math
import random
import shutil
import time
from collections import defaultdict
from datetime import datetime, timezone

DEPENDENCIES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'cv2': 'opencv-python-headless',
    'matplotlib': 'matplotlib',
    'yaml': 'pyyaml',
}
missing = [package for module, package in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing libraries:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import yaml
print('Libraries imported successfully.')

In [ ]:
PROVINCES = ['皖','沪','津','渝','冀','晋','蒙','辽','吉','黑','苏','浙','京','闽','赣','鲁','豫','鄂','湘','粤','桂','琼','川','贵','云','藏','陕','甘','青','宁','新','警','学','O']
ALPHABETS = list('ABCDEFGHJKLMNPQRSTUVWXYZ') + ['O']
ADS = list('ABCDEFGHJKLMNPQRSTUVWXYZ0123456789') + ['O']

def decode_plate(encoded):
    """Convert CCPD character indices into a plate string."""
    idx = [int(x) for x in encoded.split('_')]
    if len(idx) not in (7, 8):
        raise ValueError(f'Unexpected plate length: {len(idx)}')
    dictionaries = [PROVINCES, ALPHABETS] + [ADS] * (len(idx)-2)
    if any(not (0 <= i < len(dic)) for i, dic in zip(idx, dictionaries)):
        raise ValueError('character index out of range')
    return ''.join(dic[i] for i, dic in zip(idx, dictionaries))

def parse_ccpd_filename(path):
    """Parse bbox, tilt metadata, and text without opening the image."""
    parts = Path(path).stem.split('-')
    if len(parts) != 7:
        raise ValueError('Expected seven filename sections')

    angle = [int(v) for v in parts[1].split('_')]
    if len(angle) != 2:
        raise ValueError('Expected two tilt angles')

    coords = parts[2].split('_')
    if len(coords) != 2:
        raise ValueError('Expected two bbox endpoints')
    x1, y1 = (int(v) for v in coords[0].split('&'))
    x2, y2 = (int(v) for v in coords[1].split('&'))
    x1, x2 = sorted((x1, x2))
    y1, y2 = sorted((y1, y2))
    if x2 <= x1 or y2 <= y1:
        raise ValueError('Zero-size bbox')

    plate = decode_plate(parts[4])
    return {'x1': x1,'y1':y1,'x2':x2,'y2':y2,
            'plate_height':y2-y1,'plate_width':x2-x1,
            'tilt_horizontal':angle[0],'tilt_vertical':angle[1],
            'tilt_magnitude':max(abs(angle[0]),abs(angle[1])),
            'plate_text':plate}

EXAMPLE = '025-95_113-154&383_386&473-386&473_177&454_154&383_363&402-0_0_22_27_27_33_16-37-15.jpg'
p = parse_ccpd_filename(EXAMPLE)

assert (p['x1'],p['y1'],p['x2'],p['y2']) == (154,383,386,473)
assert p['plate_text'] == '皖AY339S'  # Check the official CCPD character-index mapping.
assert p['plate_height'] == 90
print('Parser checks passed:', p['plate_text'], 'bbox=', (p['x1'],p['y1'],p['x2'],p['y2']))

## Step 3 — Scan CCPD and select representative originals

We first read metadata from the filenames, then select a **stratified sample** based on:
- license-plate height (a measure of image scale), and
- plate tilt magnitude (a proxy for viewing angle).

The subset should reflect the source distribution rather than contain only easy-to-read plates. The **scanning and hashing** steps show progress bars.

In [ ]:
def scan_ccpd(root):
    if not root.exists():
        raise FileNotFoundError(f'CCPD_ROOT not found: {root}. Download/extract CCPD2019 and check the configuration cell.')
    paths = sorted((p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.png')))

    if not paths:
        raise FileNotFoundError(f'No JPG/PNG images found in {root}')
    records, invalid = [], []
    for image_path in tqdm(paths, desc='Parsing filename annotations', unit='image'):
        try:
            ann = parse_ccpd_filename(image_path)
            rel = image_path.relative_to(root).as_posix()
            records.append({'source_relpath':rel, 'source_path':str(image_path), **ann})
        except (ValueError, TypeError) as e:
            if len(invalid) < 15:
                invalid.append((image_path.name, str(e)))

    frame = pd.DataFrame.from_records(records)
    if frame.empty:
        raise RuntimeError('No valid filename annotations. Check the CCPD edition and filename format.')
    print(f'Images found: {len(paths):,}; valid filenames: {len(frame):,}; skipped: {len(paths)-len(frame):,}')
    if invalid:
        print('Examples of invalid filenames:', invalid[:3])
    return frame

def add_strata(df):
    result = df.copy()

    # Ranking before qcut handles repeated plate heights and tilt values.
    for col, key in [('plate_height','height_group'), ('tilt_magnitude','tilt_group')]:
        unique = int(result[col].nunique())
        n_bins = max(1, min(4, unique))
        if n_bins == 1:
            result[key] = 0
        else:
            result[key] = pd.qcut(result[col].rank(method='first'), q=n_bins, labels=False).astype(int)
    result['stratum'] = result['height_group'].astype(str) + '_' + result['tilt_group'].astype(str)
    return result

def largest_remainder(weights, total, capacities=None):
    """Allocate integer quotas totaling `total`; weights need not sum to one."""
    weights = np.asarray(weights, dtype=float)
    if capacities is None:
        capacities = np.full(len(weights), total, dtype=int)
    capacities = np.asarray(capacities, dtype=int)
    assert weights.sum() > 0 and total <= capacities.sum()

    ideal = total * weights / weights.sum()
    alloc = np.minimum(np.floor(ideal).astype(int), capacities)
    while alloc.sum() < total:
        allowed = np.flatnonzero(alloc < capacities)
        winner = max(allowed, key=lambda j: (ideal[j] - alloc[j], -j))
        alloc[winner] += 1
    return alloc

def proportional_sample(frame, n, seed):
    if len(frame) < n:
        raise ValueError(f'Requested {n} images, but only {len(frame)} valid records are available. Reduce SAMPLE_N or download full CCPD2019.')
    groups = [(key, group) for key, group in frame.groupby('stratum', sort=True)]

    count = np.array([len(g) for _, g in groups])
    quota = largest_remainder(count, n, count)

    rng = np.random.default_rng(seed)
    indices = []
    for (_, group), amount in zip(groups, quota):
        indices.extend(rng.choice(group.index.to_numpy(), size=int(amount), replace=False).tolist())
    return frame.loc[indices].sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
candidates = add_strata(scan_ccpd(CCPD_ROOT))
selected = proportional_sample(candidates, SAMPLE_N, RANDOM_SEED)
assert len(selected) == SAMPLE_N

In [ ]:
# Detect byte-for-byte duplicates even when filenames or paths differ.
# Hash only the selected subset instead of the entire source dataset.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()
selected['source_sha256'] = [sha256_file(p) for p in tqdm(selected['source_path'], desc='Checking duplicates', unit='image')]

assert selected['source_sha256'].eq(selected['source_relpath'].map(expected_hashes)).all(), 'Source checksum mismatch'

duplicates = selected.loc[selected['source_sha256'].duplicated(keep=False),'source_relpath']
if not duplicates.empty:
    raise RuntimeError(f'Byte-identical source files found: {duplicates.head(4).to_list()}. Remove them or resample; splitting was stopped to avoid data leakage.')
print('Subset selected:', len(selected), '| byte-identical duplicates: 0')
print(selected[['plate_height','tilt_magnitude','plate_text','stratum']].head(3).to_string(index=False))

## Step 4 — Create train / validation / test splits

The split is assigned to each **original image before any resizing**. This is the main defense against *train/test leakage*: versions of the same photo always belong to the same split.

Default for 5,000 originals: **3,500 train / 700 validation / 800 test**. The same split will later be applied to every generated scale.

In [ ]:
def stratified_split_exact(df, ratios, seed):
    names = list(ratios)
    targets = largest_remainder(list(ratios.values()), len(df))

    rng = np.random.default_rng(seed)
    buckets = {name: [] for name in names}
    for _, group in df.groupby('stratum', sort=True):
        ix = rng.permutation(group.index.to_numpy())
        local_counts = largest_remainder(list(ratios.values()), len(ix))
        cursor = 0
        for name, count in zip(names, local_counts):
            buckets[name].extend(ix[cursor:cursor+count].tolist())
            cursor += count
    # Reconcile rounding differences between per-stratum and global quotas.
    for _ in range(len(df) + 1):
        diff = {name: int(targets[j] - len(buckets[name])) for j,name in enumerate(names)}
        donors = [name for name in names if diff[name] < 0]
        receivers = [name for name in names if diff[name] > 0]
        if not donors and not receivers:
            break
        if not donors or not receivers:
            raise AssertionError('Split quota mismatch')
        donor = min(donors, key=lambda s:diff[s])
        receiver = max(receivers, key=lambda s:diff[s])
        transferred = buckets[donor].pop()
        buckets[receiver].append(transferred)
    else:
        raise AssertionError('Could not assign all split quotas')

    result = df.copy()
    result['split'] = ''
    for name in names:
        result.loc[buckets[name], 'split'] = name

    assert (result['split'] != '').all()
    assert result['source_relpath'].is_unique
    for name, target in zip(names, targets):
        assert (result['split'] == name).sum() == target
    return result

selected = stratified_split_exact(selected, SPLIT_RATIOS, RANDOM_SEED)
counts = selected['split'].value_counts().reindex(SPLIT_RATIOS).fillna(0).astype(int)
print('Train/val/test:', counts.to_dict())
assert int(counts.sum()) == SAMPLE_N

## Step 5 — Validate boxes and define size buckets

We open each selected image and confirm that its license-plate box has valid pixel coordinates. We then define **small / medium / large** plate categories using height thresholds calculated **only on training originals**.

The height thresholds are dataset-dependent; the project plan did not prescribe exact pixel cutoffs.

In [ ]:
def read_image_checked(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None or image.size == 0:
        raise ValueError(f'Image is unreadable: {path}')
    return image

def checked_bbox(row, width, height):
    x1,x2 = sorted((int(row['x1']),int(row['x2'])))
    y1,y2 = sorted((int(row['y1']),int(row['y2'])))
    x1,x2 = max(0,x1),min(width,x2)
    y1,y2 = max(0,y1),min(height,y2)
    if x2-x1 < 2 or y2-y1 < 2:
        raise ValueError(f'Invalid/out-of-bounds bbox for {row["source_relpath"]}')
    return (x1,y1,x2,y2)

def bucket_for_height(h, q1, q2):
    return 'small' if h <= q1 else 'medium' if h <= q2 else 'large'

def yolo_line(box, width, height):
    x1,y1,x2,y2 = box
    cx = (x1+x2)/(2*width)
    cy = (y1+y2)/(2*height)
    bw = (x2-x1)/width
    bh = (y2-y1)/height
    assert 0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1
    return f'0 {cx:.8f} {cy:.8f} {bw:.8f} {bh:.8f}\n'

# Validate bounding boxes against actual image dimensions.
widths, heights = [], []
for r in tqdm(selected.itertuples(index=False), total=len(selected), desc='Validating bounding boxes', unit='image'):
    im = read_image_checked(r.source_path)
    h,w = im.shape[:2]
    checked_bbox(r._asdict(), w, h)
    widths.append(w); heights.append(h)

selected['orig_width'] = widths
selected['orig_height'] = heights

train_heights = selected.loc[selected['split'] == 'train','plate_height']
q1, q2 = map(float, train_heights.quantile([1/3,2/3]).to_list())
selected['original_bucket'] = selected['plate_height'].apply(lambda h:bucket_for_height(h,q1,q2))
selected['parent_id'] = selected['source_relpath'].apply(lambda x:hashlib.sha256(x.encode('utf-8')).hexdigest()[:16])
assert selected['parent_id'].is_unique  # No duplicate paths or parent-ID collisions.
print('Height bucket thresholds (train only):', {'small_up_to':q1, 'medium_up_to':q2})
print('Original image buckets:',selected['original_bucket'].value_counts().to_dict())

## Step 6 — Export detection and OCR training files

For every original photo, we save each image scale and its corresponding labels:

- `images/{train,val,test}/` — full images (original and downscaled).
- `labels/{train,val,test}/` — one normalized **YOLO box** (`class_id x_center y_center width height`).
- `crops_gt/{train,val,test}/` — crops from the ground-truth box (used for **OCR-only** evaluation).
- CSV metadata and plate strings used by recognition and sensitivity experiments.

The progress bar counts processed **originals**, and each original produces `len(SCALES)` exported versions.

In [ ]:
# Preserve every existing output folder.
if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
    raise FileExistsError('Output folder is not empty; rerun Step 0 to choose a new numbered folder.')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)

variant_records, ocr_records = [], []
start = time.time()
for count,r in enumerate(tqdm(selected.to_dict('records'), desc='Exporting images and labels', unit='original'), start=1):
    image = read_image_checked(r['source_path'])
    oh,ow = image.shape[:2]
    box = checked_bbox(r, ow, oh)
    for scale in SCALES:
        tag = f's{int(round(scale*100)):03d}'
        stem = f'{r["parent_id"]}_{tag}'
        if scale == 1.0:
            scaled = image
        else:
            nw,nh = max(1,int(round(ow*scale))),max(1,int(round(oh*scale)))
            scaled = cv2.resize(image,(nw,nh),interpolation=cv2.INTER_AREA)

        h,w = scaled.shape[:2]
        fx,fy = w/ow,h/oh
        bx = (max(0,int(math.floor(box[0]*fx))),max(0,int(math.floor(box[1]*fy))),
              min(w,int(math.ceil(box[2]*fx))),min(h,int(math.ceil(box[3]*fy))))
        if bx[2] - bx[0] < 2 or bx[3] - bx[1] < 2:
            raise ValueError(f'Scaled bbox too small for {stem}')

        split = r['split']
        image_rel = Path('images')/split/f'{stem}.jpg'
        label_rel = Path('labels')/split/f'{stem}.txt'
        crop_rel = Path('crops_gt')/split/f'{stem}.png'
        for rel in (image_rel,label_rel,crop_rel):
            (OUTPUT_ROOT/rel).parent.mkdir(parents=True, exist_ok=True)

        if not cv2.imwrite(str(OUTPUT_ROOT/image_rel),scaled,[int(cv2.IMWRITE_JPEG_QUALITY),95]):
            raise IOError(f'Could not write {image_rel}')
        (OUTPUT_ROOT/label_rel).write_text(yolo_line(bx,w,h), encoding='utf-8')

        crop = scaled[bx[1]:bx[3],bx[0]:bx[2]]
        if crop.size == 0 or not cv2.imwrite(str(OUTPUT_ROOT/crop_rel),crop):
            raise IOError(f'Could not write crop {crop_rel}')

        row = {
            'image_path':image_rel.as_posix(),'label_path':label_rel.as_posix(),
            'crop_path':crop_rel.as_posix(),'source_relpath':r['source_relpath'],
            'split':split,'parent_id':r['parent_id'],'scale':scale,
            'width':w,'height':h,'x1':bx[0],'y1':bx[1],'x2':bx[2],'y2':bx[3],
            'plate_pixel_height':bx[3]-bx[1],
            'size_bucket':bucket_for_height(bx[3]-bx[1],q1,q2),
            'original_bucket':r['original_bucket'],
            'tilt_horizontal':r['tilt_horizontal'],'tilt_vertical':r['tilt_vertical'],
            'tilt_magnitude':r['tilt_magnitude'],'plate_text':r['plate_text']}
        variant_records.append(row)
        ocr_records.append({'crop_path':crop_rel.as_posix(),'plate_text':r['plate_text'],
                            'split':split,'parent_id':r['parent_id'],'scale':scale,
                            'size_bucket':row['size_bucket']})

variants = pd.DataFrame(variant_records)
ocr_manifest = pd.DataFrame(ocr_records)

selected.drop(columns=['source_path']).to_csv(OUTPUT_ROOT/'original_metadata.csv',index=False,encoding='utf-8-sig')
variants.to_csv(OUTPUT_ROOT/'metadata.csv',index=False,encoding='utf-8-sig')
ocr_manifest.to_csv(OUTPUT_ROOT/'ocr_labels.csv',index=False,encoding='utf-8-sig')

for split in SPLIT_RATIOS:
    ocr_manifest.loc[ocr_manifest['split']==split].to_csv(OUTPUT_ROOT/f'ocr_{split}.csv',index=False,encoding='utf-8-sig')
    selected.loc[selected['split']==split,'source_relpath'].to_csv(OUTPUT_ROOT/f'{split}.txt',header=False,index=False)

print(f'Done: {len(variants):,} variants ({len(selected)} originals x {len(SCALES)} scales).')

## Step 7 — OCR preprocessing (Experiment E3)

We implement exactly the three preprocessing conditions described in the project plan:

| Mode | Effect |
|:---|:---|
| `none` | Unchanged crop (baseline) |
| `gray_contrast` | Grayscale + local contrast enhancement (CLAHE) |
| `canny` | Grayscale edge detection |

The exported `preprocessing.py` can be imported directly by the teammate training the CRNN. **Agree on input channels and size** before running the OCR experiments. The notebook checks that all three modes produce valid images.

In [ ]:
MODULE_TEXT = """# CCPD OCR preprocessing: none, gray+contrast (CLAHE), and Canny.
import cv2
import numpy as np


def preprocess_plate(image, mode='none', rgb3=False):
    # Input: BGR uint8 HxWx3 or grayscale uint8. Output: uint8, BGR/grayscale by default; RGB 3ch if rgb3=True.
    if image is None or image.size == 0:
        raise ValueError('Empty plate image')
    if image.dtype != np.uint8:
        raise TypeError('Expected uint8 image')
    if image.ndim == 2:
        gray = image
        bgr = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    elif image.ndim == 3 and image.shape[2] == 3:
        bgr = image
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    else:
        raise ValueError('Expected grayscale or BGR 3-channel image')

    if mode == 'none':
        out = bgr.copy() if image.ndim == 3 else gray.copy()
    elif mode == 'gray_contrast':
        out = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)
    elif mode == 'canny':
        out = cv2.Canny(gray, threshold1=100, threshold2=200)
    else:
        raise ValueError('Unknown mode: '+str(mode))

    if rgb3:
        if out.ndim == 2:
            return cv2.cvtColor(out, cv2.COLOR_GRAY2RGB)
        return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)
    return out
"""
from preprocessing import preprocess_plate  # type: ignore
img = np.full((32,64,3),200,np.uint8)
assert preprocess_plate(img,'none',rgb3=True).shape == (32,64,3)


## Step 8 — Quality assurance and figures

We verify that all variants of one source image stay in the same split, that each YOLO label is normalized and that the generated files exist. We also save:

- original-vs-selected height and tilt distributions,
- a sample detection box and plate crop,
- side-by-side views of all three preprocessing modes.

These are **data preparation checks**, not measured YOLO/OCR model accuracy.

In [ ]:
# Leakage check: every parent_id belongs to exactly one split.
unique_split_counts = variants.groupby('parent_id')['split'].nunique()
assert unique_split_counts.eq(1).all(), 'DATA LEAKAGE: one original image appears in multiple splits!'
assert variants['image_path'].is_unique and variants['crop_path'].is_unique
assert len(variants) == SAMPLE_N*len(SCALES)
assert len(ocr_manifest) == len(variants)

for row in tqdm(variants.itertuples(index=False), total=len(variants), desc='Checking exported files', unit='variant'):
    assert (OUTPUT_ROOT/row.image_path).is_file()
    assert (OUTPUT_ROOT/row.label_path).is_file()
    assert (OUTPUT_ROOT/row.crop_path).is_file()
    label = (OUTPUT_ROOT/row.label_path).read_text(encoding='utf-8').split()
    assert len(label) == 5 and label[0] == '0'
    box = np.array([float(x) for x in label[1:]])
    assert ((box >= 0) & (box <= 1)).all()
    assert box[2] > 0 and box[3] > 0
assert set(variants['split']) == set(SPLIT_RATIOS)

# Minimal Ultralytics/YOLOv11 configuration; each split points to images/<split>.
dataset_yaml = {
    'path': str(OUTPUT_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc':1,
    'names': {0:'license_plate'}
}
(OUTPUT_ROOT/'dataset.yaml').write_text(yaml.safe_dump(dataset_yaml,allow_unicode=True,sort_keys=False),encoding='utf-8')

config_out = {'source_dataset':str(CCPD_ROOT),'created_at_utc':datetime.now(timezone.utc).isoformat(),
              'seed':RANDOM_SEED,'n_originals':SAMPLE_N,'split_ratios':SPLIT_RATIOS,
              'n_splits_originals':counts.to_dict(),'scales':list(SCALES),
              'bucket_train_height_thresholds':{'q33':q1,'q67':q2},
              'n_variants':len(variants)}
(OUTPUT_ROOT/'run_config.json').write_text(json.dumps(config_out,ensure_ascii=False,indent=2),encoding='utf-8')

# Save exact versions of the environment used for this successful run.
import importlib.metadata
runtime_packages = ('numpy', 'pandas', 'opencv-python-headless', 'matplotlib', 'PyYAML', 'tqdm', 'ipykernel', 'nbformat', 'nbclient')
requirements = '\n'.join(f'{package}=={importlib.metadata.version(package)}' for package in runtime_packages) + '\n'
(OUTPUT_ROOT/'requirements.txt').write_text(requirements, encoding='utf-8')
config_out['environment'] = {'python':sys.version, 'packages': {p:importlib.metadata.version(p) for p in runtime_packages}}
config_out['data_export_seconds'] = time.time() - start
(OUTPUT_ROOT/'run_config.json').write_text(json.dumps(config_out,ensure_ascii=False,indent=2),encoding='utf-8')

# Original-resolution evaluation is separate from the scale sensitivity dataset.
original_lists = OUTPUT_ROOT/'lists_original'
original_lists.mkdir(exist_ok=True)
for split in SPLIT_RATIOS:
    original_paths = variants.loc[(variants['split']==split) & (variants['scale']==1.0), 'image_path']
    (original_lists/f'{split}.txt').write_text('\n'.join(str((OUTPUT_ROOT/p).resolve()) for p in original_paths)+'\n', encoding='utf-8')

original_yaml = dict(dataset_yaml)
for split in SPLIT_RATIOS:
    original_yaml[split] = f'lists_original/{split}.txt'

(OUTPUT_ROOT/'dataset_original.yaml').write_text(yaml.safe_dump(original_yaml,allow_unicode=True,sort_keys=False),encoding='utf-8')
shutil.copy2(WORK_DIR/'configure_paths.py', OUTPUT_ROOT/'configure_paths.py')


In [ ]:
# Use the complete archive metadata to assess the extracted 5,000-image subset.
archive_metadata_path = DATASET_DIR.parent / 'archive_metadata.csv'
if archive_metadata_path.is_file():
    population = pd.read_csv(archive_metadata_path)
    population['tilt_magnitude'] = population[['tilt_horizontal', 'tilt_vertical']].abs().max(axis=1)
    population_label = 'CCPD eligible population'
else:
    population = candidates
    population_label = 'Input candidate population'

height_bins = np.histogram_bin_edges(population['plate_height'], bins=30)
tilt_bins = np.histogram_bin_edges(population['tilt_magnitude'], bins=30)
fig,ax = plt.subplots(1,2,figsize=(11,4))
ax[0].hist(population['plate_height'],bins=height_bins,density=True,alpha=.45,label=population_label)
ax[0].hist(selected['plate_height'],bins=height_bins,density=True,alpha=.45,label='selected originals')
ax[0].set(xlabel='Plate height (px)',ylabel='Density',title='Size distribution')
ax[0].legend(fontsize=8)
ax[1].hist(population['tilt_magnitude'],bins=tilt_bins,density=True,alpha=.45,label=population_label)
ax[1].hist(selected['tilt_magnitude'],bins=tilt_bins,density=True,alpha=.45,label='selected originals')
ax[1].set(xlabel='Tilt proxy (max absolute tilt)',ylabel='Density',title='Tilt distribution')
ax[1].legend(fontsize=8)
fig.tight_layout()
(OUTPUT_ROOT/'plots').mkdir(exist_ok=True)
fig.savefig(OUTPUT_ROOT/'plots'/'sampling_distributions.png',dpi=150)
plt.show()

show_row = variants.loc[(variants['split']=='train') & (variants['scale']==1.0)].iloc[0]
view = read_image_checked(OUTPUT_ROOT/show_row['image_path'])
colored = view.copy()
cv2.rectangle(colored,(int(show_row['x1']),int(show_row['y1'])),(int(show_row['x2']),int(show_row['y2'])),(0,255,0),2)

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].imshow(cv2.cvtColor(colored,cv2.COLOR_BGR2RGB)); ax[0].set_title('BBox ground truth');ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(read_image_checked(OUTPUT_ROOT/show_row['crop_path']),cv2.COLOR_BGR2RGB));ax[1].set_title('Ground-truth crop');ax[1].axis('off')
fig.tight_layout();fig.savefig(OUTPUT_ROOT/'plots'/'bbox_and_crop.png',dpi=150);plt.show()

crop = read_image_checked(OUTPUT_ROOT/show_row['crop_path'])
fig,axes=plt.subplots(1,3,figsize=(12,3))
for ax,mode in zip(axes,('none','gray_contrast','canny')):
    processed = preprocess_plate(crop,mode,rgb3=True)
    ax.imshow(processed);ax.set_title(mode);ax.axis('off')
fig.tight_layout();fig.savefig(OUTPUT_ROOT/'plots'/'preprocessing_modes.png',dpi=150);plt.show()
print('All checks passed. Dataset YAML:',OUTPUT_ROOT/'dataset.yaml')

## Step 9 — Save teammate handoff notes

The README explains the data format, split policy, known limitations, and which files the **YOLO detection**, **OCR recognition**, and **pipeline/evaluation** teammates should use. Share the entire `ccpd_dataset/` folder alongside this notebook. Review the team decisions at the bottom before reporting final results.

In [ ]:
readme = f"""# CCPD: prepared data (Data & Preprocessing)

Generated from CCPD original image names. Original CCPD source: https://github.com/detectRecog/CCPD

## Counts
- Original images: {SAMPLE_N}
- Splits (originals): {counts.to_dict()}
- Image scale variants: {list(SCALES)}
- Exported images / YOLO txt labels / GT crops: {len(variants)} each
- Reproducibility seed: {RANDOM_SEED}
- Thresholds for pixel-height buckets (trained only on train split): small <= {q1:.2f}; medium <= {q2:.2f}; large > {q2:.2f}

## Files
- images/{{train,val,test}}/*.jpg — source-resolution + resized images
- labels/{{train,val,test}}/*.txt — YOLO single-class normalized bbox (class 0)
- dataset.yaml — all three scales; use for explicit scale experiments
- dataset_original.yaml — only scale=1.0 for primary YOLO training/evaluation
- configure_paths.py — run once after moving this result folder: `python configure_paths.py`
- lists_original/*.txt — absolute paths to original-resolution exported images; regenerate after relocation
- crops_gt/{{train,val,test}}/*.png — crops based on GT rectangles for OCR-only evaluation
- ocr_{{train,val,test}}.csv — OCR targets and crop filenames, includes scale, parent_id, bucket
- metadata.csv — per-variant size, tilt, bbox and OCR ground truth
- original_metadata.csv — per original annotation and split
- train.txt, val.txt, test.txt — **original** source relative paths (not variant files)
- preprocessing.py — preprocess_plate(image, mode, rgb3=False)
- plots/*.png — distribution plots and examples
- run_config.json — parameters used in this run

## Reproduction
1. Open Data_Preprocessing.ipynb in the complete CCPD_Data_Preprocessing folder; the selected images and archive metadata are already included in source_data/.
2. Open this notebook and optionally edit the values in Step 0.
3. Install dependencies with `pip install -r requirements.txt` or let the notebook install missing packages, then Run All.
4. Keep a record of CCPD edition/source and parameters.

## Notes
- This re-samples a new split rather than using the official benchmark split. Results are comparable only with models using this same split.
- Splits are assigned before generating resize variants. Byte-identical source duplicates are rejected; near-duplicate photos with different JPEG encodings are not detected automatically.
- Tilt proxy = max(|horizontal tilt|, |vertical tilt|), computed from the CCPD filename.
- `preprocessing.py` returns BGR/grayscale by default, RGB with rgb3=True. OCR model author should agree on channel convention, input size, and mode.
- No model training or evaluation is performed by this notebook.
- Detection teammate: use `dataset_original.yaml` for the primary original-image experiment; `dataset.yaml` includes all three scales and changes the training distribution.
- OCR teammate: use `crops_gt/` and `ocr_train.csv`, `ocr_val.csv`, `ocr_test.csv` plus `preprocessing.py`.
- Pipeline/evaluation teammate: use `metadata.csv` to group by plate size and tilt; GT crops are for OCR-only evaluation.
"""
(OUTPUT_ROOT/'README.md').write_text(readme,encoding='utf-8')
print('Generated artifacts:')
print('\n'.join(' - '+p.name for p in sorted(OUTPUT_ROOT.iterdir())))
print('\nData preparation is complete after all cells have run successfully on real CCPD data.')

## Finish — What to check with the team

1. **Split counts:** the project plan gives 3,503 / 704 / 777 = 4,984, but requests 5,000. We use 3,500 / 700 / 800 as an explicit alternative, pending team agreement.
2. **Dataset edition and evaluation:** this pipeline assumes filename-labeled CCPD2019, with a new sample and split, not CCPD's official benchmark split.
3. **Size bins:** `small/medium/large` use training-image height tertiles; thresholds were not specified in the PDFs.
4. **OCR input:** choose whether the CRNN receives grayscale or 3-channel RGB and agree on resizing/normalization.
5. **Model results:** YOLO detection metrics, OCR accuracy, end-to-end tests, and failure analysis are separate teammates' responsibilities.

**References:** [official CCPD repository](https://github.com/detectRecog/CCPD) and the two course PDFs (`CV_Plan`, `Computer_Vision_Project_Guidelines`).